# 🥜 견과 한 줌 — 내 모델로 휴대폰에서 테스트하기

설치 없이 이 노트북만으로 **내가 학습한 `best.pt`** 를 앱에 넣고 휴대폰으로 테스트합니다.

**사용법:** 위에서부터 셀 왼쪽의 ▶ 버튼을 하나씩 누르세요. (GPU 필요 없음)

1. 앱 설치
2. `best.pt` 올리기 → 클래스 이름 확인
3. (선택) 휴대폰 안에서 인식용 모델 만들기
4. 앱 서버 켜기
5. 휴대폰 접속 주소 + QR 코드 받기 → 휴대폰으로 찍어서 테스트
6. 저장한 테스트 사진 백업

⚠️ Colab 을 닫거나 오래 두면 서버가 꺼지고, 저장한 사진도 사라집니다. 끝나기 전에 6번으로 백업하세요.

In [ ]:
# 1. 앱 설치 (1~2분)
!rm -rf /content/nuts
!git clone -q -b claude/nut-recognition-calorie-app-y8lu3b https://github.com/danielso605/-.git /content/nuts 2>/dev/null || git clone -q https://github.com/danielso605/-.git /content/nuts
%cd /content/nuts
!pip install -q -r backend/requirements.txt ultralytics qrcode
print("✅ 설치 완료")

In [ ]:
# 2. best.pt 올리기 — 실행하면 '파일 선택' 버튼이 나옵니다. 학습한 best.pt 를 고르세요.
#    (Google Drive 에 있다면 아래 DRIVE_PATH 에 경로를 적으면 업로드 대신 그 파일을 씁니다)
DRIVE_PATH = ""   # 예: "/content/drive/MyDrive/nuts/best.pt"

import os, shutil, sys
os.makedirs("/content/nuts/backend/models", exist_ok=True)
target = "/content/nuts/backend/models/nuts.pt"
if DRIVE_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(DRIVE_PATH, target)
else:
    from google.colab import files
    uploaded = files.upload()
    shutil.move(next(iter(uploaded)), target)

sys.path.insert(0, "/content/nuts/backend")
from ultralytics import YOLO
from app.detector import resolve_class
names = YOLO(target).names
print("모델 클래스 → 앱에서 쓰는 이름")
bad = []
for i, n in names.items():
    r = resolve_class(n)
    print(f"  {i}: {n:<15} → {r}")
    if r == "unknown":
        bad.append(n)
print("\n✅ 모두 인식됩니다" if not bad else f"\n⚠️ 앱이 모르는 이름: {bad} — 이 종류는 칼로리 계산에서 빠집니다. 이름을 알려 주시면 맞춰 드립니다.")

In [ ]:
# 3. (선택) 휴대폰 안에서 바로 인식용 모델 만들기 — 메인 화면에 '📱 휴대폰에서 바로 인식' 이 생깁니다
from ultralytics import YOLO
YOLO("/content/nuts/backend/models/nuts.pt").export(format="onnx", imgsz=640)
print("✅ nuts.onnx 생성")

In [ ]:
# 4. 앱 서버 켜기
#    Claude API 키가 있으면: 왼쪽 🔑(보안 비밀) 에 ANTHROPIC_API_KEY 로 저장 → 애매한 사진만 Claude 가 다시 확인(hybrid)
import os, subprocess, time, urllib.request
env = dict(os.environ, NUT_DETECTOR="yolo", YOLO_CONFIG_DIR="/tmp/Ultralytics")
try:
    from google.colab import userdata
    env["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    env["NUT_DETECTOR"] = "hybrid"
except Exception:
    pass

subprocess.run(["pkill", "-f", "uvicorn app.main:app"])
server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/nuts/backend", env=env,
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("✅ 서버 켜짐:", urllib.request.urlopen("http://localhost:8000/api/health").read().decode())
        break
    except Exception:
        time.sleep(2)
else:
    print("❌ 서버가 켜지지 않았어요. 아래 로그를 확인하세요."); print(open("/content/server.log").read()[-3000:])

In [ ]:
# 5. 휴대폰 접속 주소 만들기 (인터넷 어디서나 열리는 임시 https 주소)
import re, subprocess, time, qrcode
from IPython.display import display, HTML
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)

if url:
    display(HTML(f'''<div style="font-size:18px;line-height:1.8">
      📱 메인 앱: <a href="{url}" target="_blank">{url}</a><br>
      📸 인식 테스트: <a href="{url}/label" target="_blank">{url}/label</a><br>
      📊 정확도 리포트: <a href="{url}/report" target="_blank">{url}/report</a></div>'''))
    print("휴대폰 카메라로 QR 을 찍으면 인식 테스트 화면이 열립니다 ↓")
    display(qrcode.make(f"{url}/label").resize((260, 260)))
    print("⚠️ 이 주소를 아는 사람은 누구나 접속할 수 있어요. 테스트가 끝나면 런타임을 종료하세요.")
else:
    print("❌ 주소를 만들지 못했어요. 이 셀을 다시 실행해 보세요."); print(open("/content/tunnel.log").read()[-2000:])

### 휴대폰에서 이렇게 테스트하세요
1. QR 을 찍어 **인식 테스트(/label)** 화면을 엽니다. (처음 열 때 Cloudflare 안내가 잠깐 보일 수 있어요)
2. 📷 사진 찍기 → 모델이 찾은 박스 확인
3. 틀린 곳을 고칩니다: 박스 누르고 종류 바꾸기 · 끌어서 이동 · 모서리로 크기 · ✏️ 추가 · 🗑 삭제
4. **정확도 측정용** + 동의 체크 → 💾 저장
5. **정확도 리포트(/report)** 에서 그래프 확인 (20~30장부터 경향, 100장 이상이면 믿을 만함)

메인 앱 화면에서 브라우저 메뉴의 **'홈 화면에 추가'** 를 누르면 앱처럼 아이콘으로 열 수 있어요. (이 임시 주소는 Colab 을 끄면 바뀝니다)

In [ ]:
# 6. 저장한 테스트 사진·정답 백업 (zip 다운로드) — 다음에 다시 쓰려면 꼭 받아 두세요
import shutil, os
from google.colab import files
if os.path.exists("/content/nuts/backend/data"):
    shutil.make_archive("/content/nuts_test_data", "zip", "/content/nuts/backend/data")
    files.download("/content/nuts_test_data.zip")
else:
    print("아직 저장한 사진이 없어요.")

In [ ]:
# (참고) 이전에 받아 둔 백업을 다시 넣기 — 실행 후 4번(서버 켜기)부터 다시
from google.colab import files
import zipfile
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall("/content/nuts/backend/data")
print("✅ 복원 완료")